# 06 — The REST API

Call the FastAPI application directly. We use Starlette's in-process test client so the notebook runs without starting a server; the requests and responses are identical to a live deployment.

In [ ]:
import os, pathlib, tempfile
from meeting_memory.sdk import MeetingMemoryClient

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DB = pathlib.Path(tempfile.mkdtemp()) / "memory.db"

# Seed the database the API will serve.
MeetingMemoryClient.local(DB).import_directory(
    str(REPO / "examples" / "organizations" / "startup"), recursive=True
)
os.environ["MEETING_MEMORY_DB"] = str(DB)
print("serving:", DB)

In [ ]:
from fastapi.testclient import TestClient
from meeting_memory.api.app import app

http = TestClient(app)
print("GET /health   ->", http.get("/health").json())
print("GET /version  ->", http.get("/version").json())

In [ ]:
print(http.get("/meetings/stats").json())

In [ ]:
results = http.get("/search", params={"q": "risk", "limit": 3}).json()["results"]
for hit in results:
    print(round(hit["score"], 3), hit["memory"]["text"])

In [ ]:
report = http.get("/reports", params={"format": "markdown"}).json()
print(report["content"][:300])

## Running a live server

To serve the same API over the network (with Swagger UI at `/docs` and the dashboard at `/dashboard`):

```bash
MEETING_MEMORY_DB=atlas.db uvicorn meeting_memory.api.app:app --port 8000
```

Next: [07_deployment.ipynb](07_deployment.ipynb).